# Phase 8 — test-time augmentation, and mixing two kinds of model

Phase 7 put the working configuration (HRNet-W32 at 1024) through five-fold
cross-validation: pooled PQ 0.3807. Averaging the five fold models lifted the
leaderboard from 0.32–0.33 to 0.34, consistent with averaging removing
predictions the models are unsure of. This notebook measures two cheaper ways
to get more of the same effect. Nothing is trained.

1. **Test-time augmentation (TTA).** Each fold's model predicts its held-out
   frames in several orientations; the predictions are turned back and their
   probabilities averaged. Three variants are scored from the same forward
   passes: no TTA (identity only, which must reproduce Phase 7), the left-right
   flip pair, and all eight flips and quarter turns (the set the training
   augmentation draws from). The five folds are pooled for each variant.
2. **Mixing two kinds of model, on fold 0.** HRNet at 1024 (run *e*) is the
   better detector (RQ) and ResNet-34 at 2048 (run *f*) draws the better mask
   shapes (SQ). Their probabilities are averaged at 2048, with the rejoining
   distances doubled to mean the same distance on the Sun (48 px).

**Adoption rule** (issue #39): for TTA, pooled PQ at least +0.005 over no TTA
**and** at least four of the five folds moving the same way. For the mix, which
exists on fold 0 only, at least +0.01 over run *e*.

**Before running**, in the notebook settings:

1. Accelerator: **GPU** (one card is used; T4 or P100)
2. Internet: **on**
3. Inputs:
   - the competition data
   - the output of **`Use_HRNet` Version 3** (the five fold models, `fold0/` to `fold4/`)
   - the output of **`Filament-architecture-comparison` Version 4** (runs *e* and *f*)
4. **Save Version** with *Save output* on

Expected wall clock: about one hour (TTA on 707 frames about 35 min, the mix
about 10 min, the test submissions about 20 min).

**Output.** Scores as JSON under `tta/` and `mix/`; the test submissions
`submission_ensemble_none.csv` (Phase 7's ensemble again),
`submission_ensemble_flip.csv`, `submission_ensemble_d4.csv`, and
`submission.csv`, a copy of the variant the adoption rule picked. No probability
maps are written: Kaggle keeps at most 500 output files.

**Re-running after a failure.** Attach this notebook's earlier version as an
input too. Folds already scored are copied back and skipped.

## 1. Clone the repository and put it on the path

Cloned into `/tmp`, not `/kaggle/working`, so the clone does not count against
the 500 output files.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase8-tta-mix"  # branch or commit hash
CHECKOUT = "/tmp/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q "segmentation-models-pytorch>=0.5"

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

print("filament", filament.__version__)
print("smp", smp.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count())

## 2. Point the package at the competition data

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

os.environ["MAGFILO_ROOT"] = str(candidates[0].parent.parent)
paths = load_paths().require_dataset()
print("MAGFILO_ROOT =", os.environ["MAGFILO_ROOT"])
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))
print("test images: ", len(list(paths.test_images.glob("*.jpeg"))))

## 3. The trained models, from the attached outputs

A checkpoint is picked by the configuration stored inside it, not by its folder
name alone, so that a stray folder of the same name in another input cannot
slip in a model of the wrong fold or resolution.

In [ ]:
import json
import shutil

from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.training.loop import load_checkpoint

FOLDS = [0, 1, 2, 3, 4]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


def find_checkpoint(folder, fold, image_size, encoder_part):
    """The one best.pt under /kaggle/input/**/<folder>/ trained as described."""
    matching = []
    for candidate in sorted(Path("/kaggle/input").glob(f"**/{folder}/best.pt")):
        stored = torch.load(candidate, map_location="cpu", weights_only=False)["config"]
        if (
            int(stored.get("fold", 0)) == fold
            and int(stored["image_size"]) == image_size
            and encoder_part in str(stored["encoder"])
        ):
            matching.append(candidate)
    if not matching:
        raise SystemExit(
            f"No {folder}/best.pt with fold {fold}, {image_size}px and an encoder containing "
            f"'{encoder_part}' under /kaggle/input. Attach the inputs listed at the top."
        )
    if len(matching) > 1:
        print(f"{folder}: {len(matching)} matching checkpoints, using the first: {matching}")
    return matching[0]


fold_checkpoints = {fold: find_checkpoint(f"fold{fold}", fold, 1024, "hrnet") for fold in FOLDS}
run_e_checkpoint = find_checkpoint("e_unet_hrnet", 0, 1024, "hrnet")
run_f_checkpoint = find_checkpoint("f_unet_resnet34_2048", 0, 2048, "resnet34")
for name, checkpoint in [
    *fold_checkpoints.items(),
    ("e", run_e_checkpoint),
    ("f", run_f_checkpoint),
]:
    print(f"{name}: {checkpoint}")

dataset = load_annotations(paths.train_annotations)
val_stems = {fold: load_fold(fold, f"{CHECKOUT}/configs/splits").val for fold in FOLDS}
for fold in FOLDS:
    print(f"fold {fold}: {len(val_stems[fold])} validation frames")

# Kaggle empties /kaggle/working at the start of a session, so what an earlier
# version of this notebook finished is reachable only through the inputs.
for folder in ("tta", "mix"):
    destination = Path("/kaggle/working") / folder
    destination.mkdir(parents=True, exist_ok=True)
    for source in Path("/kaggle/input").glob(f"**/{folder}/*.json"):
        if not (destination / source.name).exists():
            shutil.copy(source, destination / source.name)
            print(f"restored {folder}/{source.name}")

## 4. TTA on every fold's held-out frames

Every frame is run once through each of the eight views; the three variants are
means over subsets of those same eight maps, so they differ in the averaging
and in nothing else. Each variant then goes through the post-processing Phase 3
settled on (threshold 0.5, minimum area 400, rejoining within 24 px at 1024),
the same chain Phase 7 scored with.

In [ ]:
import time

import numpy as np
import pandas as pd

from filament.data.disk import detect_disk
from filament.data.image import load_grayscale
from filament.evaluation import DIHEDRAL_VIEWS, FLIP_VIEWS, IDENTITY, predict_probability_views
from filament.metrics.pq import PQResult, compute_pq, pool_pq
from filament.postprocess.join import DEFAULT_MAX_OFFSET
from filament.postprocess.search import Setting, predict_from_maps
from filament.submit.rle import masks_to_gt_df

VARIANTS = {"none": (IDENTITY,), "flip": FLIP_VIEWS, "d4": DIHEDRAL_VIEWS}
SCORING = {"threshold": 0.5, "min_area": 400, "join_gap": 24.0, "join_offset": DEFAULT_MAX_OFFSET}
SIZE = 1024
# Phase 7's per-fold PQ without TTA; the "none" variant has to reproduce them.
PHASE7_PQ = {0: 0.3815, 1: 0.3774, 2: 0.3808, 3: 0.3875, 4: 0.3759}


def summarise(result, predictions):
    return {
        "pq": result.pq,
        "sq": result.sq,
        "rq": result.rq,
        "tp": result.tp,
        "fp": result.fp,
        "fn": result.fn,
        "predictions": predictions,
    }


def as_pq(row):
    return PQResult(
        pq=row["pq"], sq=row["sq"], rq=row["rq"], tp=row["tp"], fp=row["fp"], fn=row["fn"]
    )


tta_scores = {}
for fold in FOLDS:
    score_path = Path(f"/kaggle/working/tta/tta_fold{fold}.json")
    if score_path.exists():
        tta_scores[fold] = json.loads(score_path.read_text())
        print(f"fold {fold}: already scored")
        continue

    model, _ = load_checkpoint(fold_checkpoints[fold], device=DEVICE)
    truth = masks_to_gt_df(dataset, val_stems[fold])
    parts = {name: [] for name in VARIANTS}
    started = time.perf_counter()
    for position, stem in enumerate(sorted(val_stems[fold]), start=1):
        frame = load_grayscale(paths.train_images / f"{stem}.jpeg")
        disk = detect_disk(frame).scaled(SIZE / frame.shape[0])
        views = predict_probability_views(model, frame, DIHEDRAL_VIEWS, SIZE, DEVICE)
        for name, subset in VARIANTS.items():
            probability = np.mean([views[view] for view in subset], axis=0)
            parts[name].append(
                predict_from_maps({stem: probability}, Setting(SCORING), {stem: disk})
            )
        if position % 40 == 0:
            print(f"fold {fold}: {position} frames, {time.perf_counter() - started:.0f}s")

    tta_scores[fold] = {}
    for name in VARIANTS:
        predicted = pd.concat(parts[name], ignore_index=True)
        tta_scores[fold][name] = summarise(compute_pq(truth, predicted), len(predicted))
    tta_scores[fold]["seconds"] = round(time.perf_counter() - started, 1)
    score_path.write_text(json.dumps(tta_scores[fold], indent=2))

    drift = tta_scores[fold]["none"]["pq"] - PHASE7_PQ[fold]
    line = "  ".join(f"{name} {tta_scores[fold][name]['pq']:.4f}" for name in VARIANTS)
    print(f"fold {fold}: {line}  ({tta_scores[fold]['seconds']:.0f}s)")
    if abs(drift) > 0.0005:
        print(f"  WARNING: no-TTA PQ is {drift:+.4f} away from Phase 7's {PHASE7_PQ[fold]}")

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

### Pooled over the five folds, and the adoption rule

A variant is adopted when its pooled PQ is at least 0.005 above no TTA **and**
at least four folds improve. Should both pass, the higher pooled PQ wins.

In [ ]:
MIN_POOLED_GAIN = 0.005
MIN_FOLDS_SAME_WAY = 4

rows = {}
pooled = {}
for name in VARIANTS:
    pooled[name] = pool_pq(as_pq(tta_scores[fold][name]) for fold in FOLDS)
    for fold in FOLDS:
        rows[(name, f"fold {fold}")] = tta_scores[fold][name]
    rows[(name, "pooled")] = summarise(
        pooled[name], sum(tta_scores[fold][name]["predictions"] for fold in FOLDS)
    )
table = pd.DataFrame(rows).T[["pq", "sq", "rq", "tp", "fp", "fn", "predictions"]]
table[["pq", "sq", "rq"]] = table[["pq", "sq", "rq"]].astype(float).round(4)

decisions = {}
for name in ("flip", "d4"):
    gain = pooled[name].pq - pooled["none"].pq
    per_fold = [tta_scores[fold][name]["pq"] - tta_scores[fold]["none"]["pq"] for fold in FOLDS]
    improved = sum(delta > 0 for delta in per_fold)
    decisions[name] = {
        "pooled_gain": gain,
        "sq_gain": pooled[name].sq - pooled["none"].sq,
        "rq_gain": pooled[name].rq - pooled["none"].rq,
        "per_fold_gain": per_fold,
        "folds_improved": improved,
        "passes": gain >= MIN_POOLED_GAIN and improved >= MIN_FOLDS_SAME_WAY,
    }
    print(
        f"{name}: pooled PQ {gain:+.4f} (SQ {decisions[name]['sq_gain']:+.4f}, "
        f"RQ {decisions[name]['rq_gain']:+.4f}), {improved}/5 folds improved, "
        f"per fold {', '.join(f'{delta:+.4f}' for delta in per_fold)} -> "
        f"{'ADOPT' if decisions[name]['passes'] else 'reject'}"
    )

passing = [name for name in decisions if decisions[name]["passes"]]
ADOPTED = max(passing, key=lambda name: pooled[name].pq) if passing else "none"
print("adopted variant:", ADOPTED)

Path("/kaggle/working/tta/tta_summary.json").write_text(
    json.dumps(
        {
            "commit": REF,
            "scoring": SCORING,
            "pooled": {name: summarise(result, None) for name, result in pooled.items()},
            "decisions": decisions,
            "adopted": ADOPTED,
        },
        indent=2,
    )
)
table

## 5. Mixing HRNet at 1024 with ResNet-34 at 2048, on fold 0

Four readings of the same 142 frames:

- **e** — run *e* alone at 1024, scored as in Phase 6b (should give 0.3815);
- **e_2048** — run *e*'s map upsampled to 2048 and scored with the doubled
  distances. The mix is scored this way, so this is the like-for-like baseline
  that separates the effect of mixing from the effect of scoring at 2048;
- **f** — run *f* alone at 2048 (Phase 6b: 0.3782);
- **mix** — the mean of *e* upsampled and *f*, at 2048.

No TTA is applied here, so the mix is compared with the models as they were
measured before.

In [ ]:
import cv2

MIX_SIZE = 2048
SCALE = MIX_SIZE / SIZE
SCORING_2048 = SCORING | {
    "join_gap": SCORING["join_gap"] * SCALE,
    "join_offset": DEFAULT_MAX_OFFSET * SCALE,
}
PHASE6B_PQ = {"e": 0.3815, "f": 0.3782}
MIN_MIX_GAIN = 0.01

mix_path = Path("/kaggle/working/mix/mix_fold0.json")
if mix_path.exists():
    mix_scores = json.loads(mix_path.read_text())
    print("mix: already scored")
else:
    run_e, _ = load_checkpoint(run_e_checkpoint, device=DEVICE)
    run_f, _ = load_checkpoint(run_f_checkpoint, device=DEVICE)
    truth = masks_to_gt_df(dataset, val_stems[0])
    readings = {"e": [], "e_2048": [], "f": [], "mix": []}
    started = time.perf_counter()
    for position, stem in enumerate(sorted(val_stems[0]), start=1):
        frame = load_grayscale(paths.train_images / f"{stem}.jpeg")
        found = detect_disk(frame)
        disk_1024 = {stem: found.scaled(SIZE / frame.shape[0])}
        disk_2048 = {stem: found.scaled(MIX_SIZE / frame.shape[0])}
        e_map = predict_probability_views(run_e, frame, (IDENTITY,), SIZE, DEVICE)[IDENTITY]
        f_map = predict_probability_views(run_f, frame, (IDENTITY,), MIX_SIZE, DEVICE)[IDENTITY]
        e_upsampled = cv2.resize(e_map, (MIX_SIZE, MIX_SIZE), interpolation=cv2.INTER_LINEAR)
        readings["e"].append(predict_from_maps({stem: e_map}, Setting(SCORING), disk_1024))
        for name, probability in [
            ("e_2048", e_upsampled),
            ("f", f_map),
            ("mix", (e_upsampled + f_map) / 2),
        ]:
            readings[name].append(
                predict_from_maps({stem: probability}, Setting(SCORING_2048), disk_2048)
            )
        if position % 40 == 0:
            print(f"mix: {position} frames, {time.perf_counter() - started:.0f}s")

    mix_scores = {}
    for name, parts in readings.items():
        predicted = pd.concat(parts, ignore_index=True)
        mix_scores[name] = summarise(compute_pq(truth, predicted), len(predicted))
    mix_path.write_text(json.dumps(mix_scores, indent=2))
    del run_e, run_f
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

for name, expected in PHASE6B_PQ.items():
    if abs(mix_scores[name]["pq"] - expected) > 0.0005:
        print(f"WARNING: {name} alone gives {mix_scores[name]['pq']:.4f}, Phase 6b gave {expected}")

for baseline in ("e", "e_2048"):
    gain = mix_scores["mix"]["pq"] - mix_scores[baseline]["pq"]
    print(
        f"mix against {baseline}: PQ {gain:+.4f}, "
        f"SQ {mix_scores['mix']['sq'] - mix_scores[baseline]['sq']:+.4f}, "
        f"RQ {mix_scores['mix']['rq'] - mix_scores[baseline]['rq']:+.4f}"
    )
mix_gain = mix_scores["mix"]["pq"] - mix_scores["e"]["pq"]
print(
    "worth training f on folds 1-4:"
    if mix_gain >= MIN_MIX_GAIN
    else "not worth training f on folds 1-4:",
    f"{mix_gain:+.4f} against the {MIN_MIX_GAIN} bar",
)
pd.DataFrame(mix_scores).T.astype(float).round(4)

## 6. The test submissions

One pass over the test frames with all five fold models and all eight views.
Each variant's map is the mean over the five models of that model's mean over
the variant's views, then the same post-processing as above. `none` is Phase
7's ensemble again (1,135 masks); the file named `submission.csv` is the
adopted variant.

The overlap check runs on every file. Kaggle rejects a submission whose masks
share a pixel, and a rejected attempt still counts against the five allowed per
day.

In [ ]:
from filament.metrics.overlap import check_no_overlap
from filament.submit.rle import write_submission

models = {fold: load_checkpoint(fold_checkpoints[fold], device=DEVICE)[0] for fold in FOLDS}
test_stems = sorted(path.stem for path in paths.test_images.glob("*.jpeg"))
test_parts = {name: [] for name in VARIANTS}
started = time.perf_counter()
for position, stem in enumerate(test_stems, start=1):
    frame = load_grayscale(paths.test_images / f"{stem}.jpeg")
    disk = {stem: detect_disk(frame).scaled(SIZE / frame.shape[0])}
    views = {
        fold: predict_probability_views(model, frame, DIHEDRAL_VIEWS, SIZE, DEVICE)
        for fold, model in models.items()
    }
    for name, subset in VARIANTS.items():
        probability = np.mean(
            [np.mean([views[fold][view] for view in subset], axis=0) for fold in models], axis=0
        )
        test_parts[name].append(predict_from_maps({stem: probability}, Setting(SCORING), disk))
    if position % 30 == 0:
        print(f"{position}/{len(test_stems)}, {time.perf_counter() - started:.0f}s")

written = {}
for name in VARIANTS:
    predicted = pd.concat(test_parts[name], ignore_index=True)
    path = Path(f"/kaggle/working/submission_ensemble_{name}.csv")
    write_submission(predicted, path)
    # Raises before the file is counted as written if any two masks touch.
    check_no_overlap(path)
    frames = predicted["filament_id"].str.rsplit("_", n=1).str[0].nunique()
    written[name] = {"masks": len(predicted), "frames_with_predictions": int(frames)}
    print(f"{path.name}: {len(predicted)} masks over {frames} of {len(test_stems)} frames")

shutil.copy(f"/kaggle/working/submission_ensemble_{ADOPTED}.csv", "/kaggle/working/submission.csv")
print(f"submission.csv = submission_ensemble_{ADOPTED}.csv")
Path("/kaggle/working/submission_info.json").write_text(
    json.dumps(
        {"commit": REF, "folds": FOLDS, "scoring": SCORING, "adopted": ADOPTED, "files": written},
        indent=2,
    )
)

In [ ]:
# Kaggle keeps at most 500 output files and drops the rest without an error.
saved = [item for item in Path("/kaggle/working").rglob("*") if item.is_file()]
print(f"{len(saved)} files in /kaggle/working")
if len(saved) > 500:
    raise SystemExit(f"{len(saved)} output files; Kaggle would silently drop some of them.")

## 7. What to record

Into the lab notebook:

- each fold's PQ, SQ, RQ, TP, FP, FN and prediction count for none / flip / d4,
  the pooled lines, and the decision with the per-fold gains
- whether "none" reproduced Phase 7 fold by fold
- the four fold-0 readings of the mix, and whether *f* is worth training on
  folds 1–4
- the mask counts of the three test submissions, and the leaderboard score of
  `submission.csv` only if a variant was adopted; the commit hash this notebook
  cloned

The leaderboard shows two decimals and the public part is about half the test
set, so differences below 0.01 there are not readable.